In [1]:
import torch

# 1. Check for CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f"CUDA is available! GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA capability: {torch.cuda.get_device_capability(0)}") # Major and minor revision numbers
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Allocated memory: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"Cached memory: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")

Using device: cuda
CUDA is available! GPU Name: NVIDIA GeForce RTX 4070 Laptop GPU
CUDA capability: (8, 9)
Number of GPUs: 1
Allocated memory: 0.00 MB
Cached memory: 0.00 MB


In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torchvision import models, transforms
import pandas as pd
import numpy as np
from PIL import Image, ImageFile
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

c:\Users\Lenovo\anaconda3\envs\research_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
try:
    multimodal_validate_df = pd.read_csv('multimodal_validate.tsv', sep='\t')
    multimodal_test_df = pd.read_csv('multimodal_test_public.tsv', sep='\t')
    multimodal_train_df = pd.read_csv('multimodal_train_sample.tsv', sep='\t')
    LOCAL_IMAGE_DIR = '.\dataset_image'
    print("TSV files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading TSV file: {e}. Please ensure the files are in the correct directory.")
    exit()

TSV files loaded successfully.


In [4]:
multimodal_train_df.head()

,author,clean_title,created_utc,domain,hasImage,id,image_url,linked_submission_id,num_comments,score,subreddit,title,upvote_ratio,2_way_label,3_way_label,6_way_label
0,Alexithymia,my walgreens offbrand mucinex was engraved wit...,1.551641e+09,i.imgur.com,True,awxhir,https://external-preview.redd.it/WylDbZrnbvZdB...,NaN,2.0,12,mildlyinteresting,My Walgreens offbrand Mucinex was engraved wit...,0.84,1,0,0
1,VIDCAs17,this concerned sink with a tiny hat,1.534727e+09,i.redd.it,True,98pbid,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,NaN,2.0,119,pareidolia,This concerned sink with a tiny hat,0.99,0,2,2
2,prometheus1123,hackers leak emails from uae ambassador to us,1.496511e+09,aljazeera.com,True,6f2cy5,https://external-preview.redd.it/6fNhdbc6K1vFA...,NaN,1.0,44,neutralnews,Hackers leak emails from UAE ambassador to US,0.92,1,0,0
3,NaN,puppy taking in the view,1.471341e+09,i.imgur.com,True,4xypkv,https://external-preview.redd.it/HLtVNhTR6wtYt...,NaN,26.0,250,photoshopbattles,PsBattle: Puppy taking in the view,0.95,1,0,0
4,3rikR3ith,i found a face in my sheet music too,1.525318e+09,i.redd.it,True,8gnet9,https://preview.redd.it/ri7ut2wn8kv01.jpg?widt...,NaN,2.0,13,pareidolia,I found a face in my sheet music too!,0.84,0,2,2


In [5]:
if '2_way_label' in multimodal_train_df.columns:
    print(f"Labels in multimodal_test_df: {multimodal_train_df['2_way_label'].value_counts()}")
else:
    print("Warning: '2_way_label' column not found in multimodal_test_df.")

Labels in multimodal_test_df: 2_way_label
0    341919
1    222081
Name: count, dtype: int64


In [6]:
# --- 2. Text Tokenization Setup (BERT) ---
# Initialize the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Define max sequence length
MAX_SEQ_LENGTH = 128

def tokenize_texts(texts, tokenizer, max_seq_length):
    text_list = texts.fillna('').tolist()
    encoded_inputs = tokenizer(
        text_list,
        max_length=max_seq_length,
        truncation=True,
        padding='max_length',
        return_tensors='pt' # Return PyTorch tensors
    )
    return encoded_inputs

# Tokenize the 'clean_title' column for the validation dataset
tokenized_validate_texts = tokenize_texts(multimodal_validate_df['clean_title'], tokenizer, MAX_SEQ_LENGTH)

# Tokenize the 'clean_title' column for the test dataset (for completeness, though not directly used in training setup below)
tokenized_test_texts = tokenize_texts(multimodal_test_df['clean_title'], tokenizer, MAX_SEQ_LENGTH)

tokenized_train_texts = tokenize_texts(multimodal_train_df['clean_title'], tokenizer, MAX_SEQ_LENGTH)


In [ ]:
# --- 3. Custom PyTorch Dataset for Multimodal Data ---
ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = 1000_000_000

# --- Image Transformations (as provided) ---
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # ImageNet normalization
])

# --- Custom PyTorch Dataset for Multimodal Data ---
class MultimodalDataset(Dataset):
    def __init__(self, dataframe, tokenized_texts, image_dir,
                 image_transform=None, labels_column='2_way_label',
                 image_id_column='id', image_extension='.jpg',
                 default_image_size=(3, 224, 224)):

        # Make a copy to avoid modifying the original dataframe passed in
        initial_dataframe = dataframe.copy()
        self.tokenized_texts = tokenized_texts
        self.image_dir = image_dir
        self.image_transform = image_transform
        self.default_image_size = default_image_size
        self.image_extension = image_extension

        if image_id_column not in initial_dataframe.columns:
            raise ValueError(f"DataFrame must contain an '{image_id_column}' column for image filenames.")

        # Create the full image filename column
        initial_dataframe['image_filename'] = initial_dataframe[image_id_column].astype(str) + self.image_extension

        # Drop rows where the image_id_column itself is NaN (should ideally be handled earlier)
        initial_dataframe.dropna(subset=[image_id_column], inplace=True)
        print(f"Initial samples after handling image filenames: {len(initial_dataframe)}")

        # --- Filter out invalid image paths and corrupted images during initialization ---
        valid_indices = []
        rows_to_drop = []
        for i, row in initial_dataframe.iterrows():
            image_path = os.path.join(self.image_dir, row['image_filename'])
            if not os.path.exists(image_path):
                print(f"Image file not found: {image_path}. Marking row for deletion.")
                rows_to_drop.append(i)
                continue # Skip to next row

            try:
                with Image.open(image_path) as img:
                    img.verify() # Verify if it's a valid image
                valid_indices.append(i)
            except (IOError, SyntaxError) as e:
                print(f"Error validating image {image_path}: {e}. Marking row for deletion.")
                rows_to_drop.append(i)
            except Exception as e:
                print(f"An unexpected error occurred while verifying {image_path}: {e}. Marking row for deletion.")
                rows_to_drop.append(i)

        self.dataframe = initial_dataframe.loc[valid_indices].reset_index(drop=True)

        self.tokenized_texts = {key: torch.tensor(val[valid_indices])
                                for key, val in tokenized_texts.items()}


        print(f"Dataset finalized. Total valid samples after image filtering: {len(self.dataframe)}")

        self.labels = self.dataframe[labels_column].values.astype(np.float32)

        if np.isnan(self.labels).any():
            nan_count = np.sum(np.isnan(self.labels))
            print(f"Warning: {nan_count} actual NaN labels found in original data. Setting them to 0.0.")
            self.labels[np.isnan(self.labels)] = 0.0

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        input_ids = self.tokenized_texts['input_ids'][idx]
        attention_mask = self.tokenized_texts['attention_mask'][idx]
        token_type_ids = self.tokenized_texts.get('token_type_ids', torch.zeros_like(input_ids, dtype=torch.long))[idx]


        image_filename = self.dataframe['image_filename'].iloc[idx]
        image_path = os.path.join(self.image_dir, image_filename)

        img = Image.open(image_path).convert('RGB')
        if self.image_transform:
            image_data = self.image_transform(img)
        else:
            image_data = transforms.ToTensor()(img)
            print(f"Warning: No image_transform provided for {image_path}. Image might not be correctly preprocessed for ResNet.")

        label = torch.tensor(self.labels[idx], dtype=torch.float32)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'token_type_ids': token_type_ids,
            'image_input': image_data,
            'label': label
        }

In [8]:
# --- 4. Image Preprocessing Transforms ---
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # Normalization using ImageNet mean and std
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [9]:
# --- 5. Multimodal Model Architecture (PyTorch) ---
class MultimodalMisinformationModel(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', num_classes=1):
        super(MultimodalMisinformationModel, self).__init__()

        # Text Stream (BERT)
        self.bert = BertModel.from_pretrained(bert_model_name)
        # Freeze BERT layers initially (optional)
        for param in self.bert.parameters():
            param.requires_grad = False
        self.bert_dropout = nn.Dropout(0.2)

        # Image Stream (ResNet50)
        self.resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove the original fully connected layer for classification
        self.resnet = nn.Sequential(*(list(self.resnet.children())[:-1])) # Remove last FC layer
        # Freeze ResNet layers initially (optional)
        for param in self.resnet.parameters():
            param.requires_grad = False
        self.resnet_dropout = nn.Dropout(0.2)

        # Fusion Layer and Classification Head
        # BERT output size is 768 for bert-base-uncased
        # ResNet50's final average pooling layer output size is 2048
        fused_feature_size = self.bert.config.hidden_size + 2048
        
        self.classifier = nn.Sequential(
            nn.Linear(fused_feature_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
            nn.Sigmoid() # For binary classification
        )

    def forward(self, input_ids, attention_mask, token_type_ids, image_input):
        # Text Forward Pass
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        text_features = bert_output.pooler_output # [CLS] token representation
        text_features = self.bert_dropout(text_features)

        # Image Forward Pass
        resnet_output = self.resnet(image_input)
        image_features = resnet_output.view(resnet_output.size(0), -1) # Flatten the features after pooling
        image_features = self.resnet_dropout(image_features)

        # Fusion
        fused_features = torch.cat((text_features, image_features), dim=1)

        # Classification
        output = self.classifier(fused_features)
        return output

In [10]:
# --- 6. Build and Prepare for Training ---
print("\nBuilding the multimodal model...")
model = MultimodalMisinformationModel(num_classes=1) # 1 for binary classification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model moved to: {device}")


Building the multimodal model...
Model moved to: cuda


In [11]:
# --- 7. Create DataLoaders ---
validate_dataset = MultimodalDataset(
    multimodal_validate_df,
    tokenized_validate_texts,
    image_dir=LOCAL_IMAGE_DIR, 
    image_transform=image_transform,
    labels_column='2_way_label',
    image_id_column='id',         # <--- Confirm this is the column name in your TSV
    image_extension='.jpg'        # <--- Confirm your image file extension
)

BATCH_SIZE = 160
validate_dataloader = DataLoader(validate_dataset, batch_size=BATCH_SIZE, shuffle=True)

train_dataset = MultimodalDataset(
    multimodal_train_df,
    tokenized_train_texts,
    image_dir=LOCAL_IMAGE_DIR,
    image_transform=image_transform,
    labels_column='2_way_label',
    image_id_column='id',
    image_extension='.jpg'
)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

Initial samples after handling image filenames: 59342
Image file not found: .\dataset_image\ds4mu90.jpg. Marking row for deletion.
Image file not found: .\dataset_image\c4dbi8e.jpg. Marking row for deletion.
Image file not found: .\dataset_image\ch1pcor.jpg. Marking row for deletion.
Image file not found: .\dataset_image\dgai382.jpg. Marking row for deletion.
Image file not found: .\dataset_image\cmk2pxb.jpg. Marking row for deletion.
Image file not found: .\dataset_image\ew160sq.jpg. Marking row for deletion.
Image file not found: .\dataset_image\c7qs76w.jpg. Marking row for deletion.
Image file not found: .\dataset_image\d6ww1kw.jpg. Marking row for deletion.
Image file not found: .\dataset_image\ctlphvf.jpg. Marking row for deletion.
Image file not found: .\dataset_image\dzs40g7.jpg. Marking row for deletion.
Image file not found: .\dataset_image\cr8n9al.jpg. Marking row for deletion.
Image file not found: .\dataset_image\c89rc9d.jpg. Marking row for deletion.
Image file not found: 

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2656\1160098689.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.tokenized_texts = {key: torch.tensor(val[valid_indices])


Dataset finalized. Total valid samples after image filtering: 59299
Initial samples after handling image filenames: 564000
Image file not found: .\dataset_image\dhiibiw.jpg. Marking row for deletion.
Image file not found: .\dataset_image\8mp3d1.jpg. Marking row for deletion.
Image file not found: .\dataset_image\d1vwukd.jpg. Marking row for deletion.
Image file not found: .\dataset_image\cjxmb4c.jpg. Marking row for deletion.
Image file not found: .\dataset_image\esi0vrr.jpg. Marking row for deletion.
Image file not found: .\dataset_image\dzp14sm.jpg. Marking row for deletion.
Image file not found: .\dataset_image\dij5nzu.jpg. Marking row for deletion.
Image file not found: .\dataset_image\d5evm0v.jpg. Marking row for deletion.
Image file not found: .\dataset_image\cqhowgl.jpg. Marking row for deletion.
Image file not found: .\dataset_image\dq4x5ki.jpg. Marking row for deletion.
Image file not found: .\dataset_image\c8knklu.jpg. Marking row for deletion.
Image file not found: .\dataset

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_2656\1160098689.py:60: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.tokenized_texts = {key: torch.tensor(val[valid_indices])


Dataset finalized. Total valid samples after image filtering: 563612


In [ ]:
# --- 8. Define Loss Function and Optimizer ---
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm

# --- 9. Training Loop ---
print("\n--- Training Section ---")

NUM_EPOCHS = 8

PATIENCE = 3
min_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared.")

for epoch in range(NUM_EPOCHS):
    model.train() # Set model to training mode
    running_loss = 0.0
    
    train_loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1} Training", leave=False)
    for i, data in enumerate(train_dataloader):
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        token_type_ids = data['token_type_ids'].to(device) # Only if your BertModel expects it
        image_input = data['image_input'].to(device)
        labels = data['label'].to(device).view(-1, 1) # Reshape labels to match output (N, 1)

        optimizer.zero_grad() # Zero the parameter gradients

        outputs = model(input_ids, attention_mask, token_type_ids, image_input)
        loss = criterion(outputs, labels)
        loss.backward() # Backpropagation
        optimizer.step() # Optimize

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_dataloader)
    print(f"Epoch {epoch+1}, Training Loss: {avg_train_loss:.4f}")

    model.eval() # Set model to evaluation mode
    val_running_loss = 0.0
    all_val_predictions = []
    all_val_labels = []

    val_loop = tqdm(validate_dataloader, desc=f"Epoch {epoch+1} Validation", leave=False)
    with torch.no_grad():
        for data in validate_dataloader:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            token_type_ids = data['token_type_ids'].to(device)
            image_input = data['image_input'].to(device)
            labels = data['label'].to(device).view(-1, 1)

            outputs = model(input_ids, attention_mask, token_type_ids, image_input)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item()

            probabilities = torch.sigmoid(outputs) # Get probabilities for metrics
            all_val_predictions.extend(probabilities.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_running_loss / len(validate_dataloader)

    # --- Calculate validation metrics for multiple thresholds ---
    best_f1_for_epoch = -1
    best_threshold_for_epoch = 0.5 # Default if no improvement
    best_metrics_for_epoch = {}

    thresholds_to_test = [0.6, 0.61, 0.62, 0.63, 0.64,0.65]

    for threshold in thresholds_to_test:
        binary_val_predictions = (np.array(all_val_predictions) > threshold).astype(int)

        val_accuracy = accuracy_score(all_val_labels, binary_val_predictions)
        val_precision = precision_score(all_val_labels, binary_val_predictions, zero_division=0)
        val_recall = recall_score(all_val_labels, binary_val_predictions, zero_division=0)
        val_f1 = f1_score(all_val_labels, binary_val_predictions, zero_division=0)

        print(f"  Threshold {threshold:.2f} -> Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

        # Keep track of the best F1 score and its corresponding threshold for this epoch
        if val_f1 > best_f1_for_epoch:
            best_f1_for_epoch = val_f1
            best_threshold_for_epoch = threshold
            best_metrics_for_epoch = {
                'accuracy': val_accuracy,
                'precision': val_precision,
                'recall': val_recall,
                'f1': val_f1
            }
    
    print(f"Epoch {epoch+1}, Best F1 (Threshold {best_threshold_for_epoch:.2f}): {best_f1_for_epoch:.4f}")

    # print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss:.4f}")
    # print(f"Epoch {epoch+1}, Validation Accuracy: {val_accuracy:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}")

    # --- Early Stopping Logic ---
    if avg_val_loss < min_val_loss:
        min_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict() # Save the model state
        print("Validation loss improved. Saving best model state.")
    else:
        epochs_no_improve += 1
        print(f"Validation loss did not improve for {epochs_no_improve} epochs.")
        if epochs_no_improve == PATIENCE:
            print(f"Early stopping triggered after {epoch + 1} epochs. No improvement for {PATIENCE} consecutive epochs.")
            break # Exit the training loop

# Load the best model state after training
if best_model_state:
    model.load_state_dict(best_model_state)
    print("Loaded best model state from training.")

print("Model training complete.")


--- Training Section ---
CUDA cache cleared.


Epoch 1, Training Loss: 0.6478


Epoch 1 Validation:   0%|          | 0/371 [00:00<?, ?it/s]


  Threshold 0.60 -> Accuracy: 0.7929, Precision: 0.7930, Recall: 0.6407, F1: 0.7087
  Threshold 0.61 -> Accuracy: 0.7913, Precision: 0.7996, Recall: 0.6265, F1: 0.7025
  Threshold 0.62 -> Accuracy: 0.7888, Precision: 0.8040, Recall: 0.6122, F1: 0.6951
  Threshold 0.63 -> Accuracy: 0.7858, Precision: 0.8086, Recall: 0.5965, F1: 0.6865
  Threshold 0.64 -> Accuracy: 0.7827, Precision: 0.8135, Recall: 0.5805, F1: 0.6775
  Threshold 0.65 -> Accuracy: 0.7795, Precision: 0.8196, Recall: 0.5633, F1: 0.6677
Epoch 1, Best F1 (Threshold 0.60): 0.7087
Validation loss improved. Saving best model state.



c:\Users\Lenovo\anaconda3\envs\research_env\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

  Threshold 0.60 -> Accuracy: 0.7949, Precision: 0.8032, Recall: 0.6339, F1: 0.7085
  Threshold 0.61 -> Accuracy: 0.7924, Precision: 0.8075, Recall: 0.6201, F1: 0.7015
  Threshold 0.62 -> Accuracy: 0.7899, Precision: 0.8127, Recall: 0.6051, F1: 0.6937
  Threshold 0.63 -> Accuracy: 0.7873, Precision: 0.8179, Recall: 0.5907, F1: 0.6860
  Threshold 0.64 -> Accuracy: 0.7834, Precision: 0.8229, Recall: 0.5725, F1: 0.6752
  Threshold 0.65 -> Accuracy: 0.7793, Precision: 0.8274, Recall: 0.5545, F1: 0.6640
Epoch 2, Best F1 (Threshold 0.60): 0.7085
Validation loss improved. Saving best model state.


Epoch 3 Training:   0%|          | 0/3523 [00:00<?, ?it/s]
c:\Users\Lenovo\anaconda3\envs\research_env\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 3, Training Loss: 0.6453


  Threshold 0.60 -> Accuracy: 0.7975, Precision: 0.8062, Recall: 0.6385, F1: 0.7126
  Threshold 0.61 -> Accuracy: 0.7956, Precision: 0.8122, Recall: 0.6247, F1: 0.7062
  Threshold 0.62 -> Accuracy: 0.7925, Precision: 0.8162, Recall: 0.6097, F1: 0.6980
  Threshold 0.63 -> Accuracy: 0.7896, Precision: 0.8211, Recall: 0.5945, F1: 0.6896
  Threshold 0.64 -> Accuracy: 0.7864, Precision: 0.8262, Recall: 0.5786, F1: 0.6806
  Threshold 0.65 -> Accuracy: 0.7831, Precision: 0.8319, Recall: 0.5621, F1: 0.6709
Epoch 3, Best F1 (Threshold 0.60): 0.7126
Validation loss improved. Saving best model state.


c:\Users\Lenovo\anaconda3\envs\research_env\lib\site-packages\PIL\Image.py:1045: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


In [ ]:
# --- 10. Evaluation and Prediction ---
print("\n--- Evaluation and Prediction Section ---")

print("\n--- Checking labels in test_dataset ---")

test_dataset = MultimodalDataset(
    multimodal_test_df,
    tokenized_test_texts,
    image_dir=LOCAL_IMAGE_DIR,
    image_transform=image_transform,
    labels_column='2_way_label'
)

if '2_way_label' in multimodal_test_df.columns:
    print(f"Labels in multimodal_test_df: {multimodal_test_df['2_way_label'].value_counts()}")
else:
    print("Warning: '2_way_label' column not found in multimodal_test_df.")

# Then, check the labels as they are processed by the DataLoader
test_labels_from_dataloader = []
# Create a temporary dataloader with a small batch size for quick iteration
temp_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False) # Now test_dataset is defined!
for i, data in enumerate(temp_dataloader):
    test_labels_from_dataloader.extend(data['label'].cpu().numpy().flatten())
    if i > 10: break # Only check a few batches to avoid long waits

print(f"Labels observed from test_dataloader batches: {np.unique(test_labels_from_dataloader, return_counts=True)}")
print("--- End of label check ---")

model.eval() # Set model to evaluation mode
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_predictions = []
all_labels = []
with torch.no_grad(): # No gradient calculations needed
    for data in test_dataloader:
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        token_type_ids = data['token_type_ids'].to(device)
        image_input = data['image_input'].to(device)
        labels = data['label'].to(device).view(-1, 1)

        outputs = model(input_ids, attention_mask, token_type_ids, image_input)
        all_predictions.extend(outputs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Convert probabilities to binary predictions
probabilities = torch.sigmoid(torch.tensor(all_predictions)).numpy() # Apply sigmoid if outputs are logits
binary_predictions = (probabilities > 0.65).astype(int)


print(f"Accuracy: {accuracy_score(all_labels, binary_predictions):.5f}")
print(f"Precision: {precision_score(all_labels, binary_predictions):.5f}")
print(f"Recall: {recall_score(all_labels, binary_predictions):.5f}")
print(f"F1 Score: {f1_score(all_labels, binary_predictions):.5f}")